In [4]:
import os
import numpy as np
import warnings 
import os
import xarray as xr
import numpy as np
import pandas as pd

from datetime import datetime

from string import Template

from experiment_configs import get_experiment_dict


In [5]:
class SurfaceEnergyBalanceProcessor:
    def __init__(self, data_dir, varnames, experiment, ensemble_list, years, outdir,
                 lat_bounds=None, lon_bounds=None, landmask_file=None,
                 expected_frequency='daily'):
        self.data_dir = data_dir
        self.varnames = varnames
        self.experiment = experiment
        self.ensemble_list = ensemble_list
        self.years = years
        self.outdir = os.path.join(outdir, experiment)
        self.lat_bounds = lat_bounds
        self.lon_bounds = lon_bounds
        self.expected_frequency = expected_frequency

        # Latent heat of vaporization (J/kg)
        self.Lv = 2.5e6

        # Load landmask if provided
        self.landmask = xr.open_dataset(landmask_file)['landmask'] if landmask_file else None
        os.makedirs(self.outdir, exist_ok=True)

    def build_paths(self, vname, ensemble):
        return [os.path.join(self.data_dir, f"{vname}.{ensemble}.{year}.nc") for year in self.years]

    def _load_variable(self, varname, ensemble):
        original_var = varname
        possible_vars = [varname]
        if varname == "LHFLX":
            possible_vars = ["LHFLX", "QFLX"]

        alt_dir = self.data_dir.replace("daily", "6hourly")

        for v in possible_vars:
            for base_dir in [self.data_dir, alt_dir]:
                files = [os.path.join(base_dir, f"{v}.{ensemble}.{year}.nc") for year in self.years]
                if all(os.path.exists(f) for f in files):
                    if base_dir == alt_dir:
                        self.expected_frequency = '6h'
                        print(f"[INFO] {v} found in 6-hourly path: resampling will be applied.")
                    datasets = [xr.open_dataset(f)[v] for f in files]
                    ds = xr.concat(datasets, dim='time')

                    if original_var == "LHFLX" and v == "QFLX":
                        print("[INFO] Converting QFLX to LHFLX using Lv.")
                        ds = ds * self.Lv
                        ds.attrs["units"] = "W/m2"
                        ds.name = "LHFLX"

                    return ds

        raise FileNotFoundError(f"[ERROR] Could not find {possible_vars} in {self.data_dir} or {alt_dir} for {ensemble} in daily or 6-hourly paths.")

    def _load_dataset(self, ensemble):
        ds = xr.Dataset({
            key: self._load_variable(vname, ensemble)
            for key, vname in self.varnames.items()
        })
        return self._resample_if_needed(ds)

    def _resample_if_needed(self, ds):
        if self.expected_frequency.lower() == '6h':
            print("[INFO] Resampling 6-hourly data to daily means...")
            return ds.resample(time='1D').mean()
        return ds

    def _subset_region(self, ds, region):
        if self.lat_bounds:
            ds = ds.sel(lat=slice(*self.lat_bounds))
        if self.lon_bounds:
            ds = ds.sel(lon=slice(*self.lon_bounds))

        if region == "land" and self.landmask is not None:
            ds = ds.where(self.landmask > 0)
        elif region == "ocean" and self.landmask is not None:
            ds = ds.where(self.landmask == 0)
        return ds

    def _compute_seb_terms(self, ds):
        le = ds['LHFLX']
        sh = ds['SHFLX']
        fsns = ds['FSNS']
        flns = ds['FLNS']
        rn = fsns - flns   #FSNS - FLNS
        residual = rn - (le + sh)

        eps = 1e-6
        bowen = xr.where(le > eps, sh / le, np.nan)

        return xr.Dataset({
            'Rn': rn,
            'LE': le,
            'SH': sh,
            'Residual': residual,
            'BowenRatio': bowen
        })

    def _save_outputs(self, ds_daily, ensemble, region):
        time_index = ds_daily.time.to_index()
        year_months = sorted(set((t.year, t.month) for t in time_index))

        ds_monthly = ds_daily.resample(time='1MS').mean()

        for year, month in year_months:
            month_mask = (ds_daily.time.dt.year == year) & (ds_daily.time.dt.month == month)
            daily_data = ds_daily.sel(time=month_mask)
            if daily_data.time.size == 0:
                continue

            ym_str = f"{year}-{month:02d}"
            daily_fname = f"seb_daily_{ensemble}_{region}_{ym_str}.nc"
            daily_path = os.path.join(self.outdir, daily_fname)
            daily_data.to_netcdf(daily_path)

            monthly_data = ds_monthly.sel(time=ds_monthly.time.dt.year == year)
            monthly_data = monthly_data.sel(time=monthly_data.time.dt.month == month)
            if monthly_data.time.size > 0:
                monthly_fname = f"seb_monthly_{ensemble}_{region}_{ym_str}.nc"
                monthly_path = os.path.join(self.outdir, monthly_fname)
                monthly_data.to_netcdf(monthly_path)

        print(f"[{self.experiment}] {ensemble} - {region.upper()}: saved monthly + per-month daily files")

    def process(self):
        for ensemble in self.ensemble_list:
            print(f"→ Processing {self.experiment} | {ensemble}")
            ds = self._load_dataset(ensemble)
            for region in ['global', 'land', 'ocean']:
                ds_region = self._subset_region(ds, region)
                ds_daily = self._compute_seb_terms(ds_region)
                self._save_outputs(ds_daily, ensemble, region)

In [6]:
if __name__ == "__main__":

    #top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    #out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    #fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"

    top_path = "/pscratch/sd/z/zhan391/e3sm_dart"
    out_path = "/pscratch/sd/z/zhan391/e3sm_dart/diag_dart"
    fig_path = "/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"
    
    landmask_file = "/pscratch/sd/z/zhan391/e3sm_dart/diag_dart/landmask_1x1.nc"
    
    exp_key = {
        "v3_spinup": 2011,
        "v3_hindcast": 2012
    } #DA period 
    #exp = 'v3_spinup'
    exp = 'v3_hindcast'
    exp_dict = get_experiment_dict(exp)
    years = [exp_key[exp]] 

    varnames = {
        'LHFLX' : 'LHFLX',
        'SHFLX' : 'SHFLX',
        'FSNS'  : 'FSNS',
        'FLNS'  : 'FLNS'
    }
    
    for exp_name, exp_meta in exp_dict.items():
        print(f"\n[INFO] Running experiment: {exp_name}")
        path = exp_meta['path']
        template_str = exp_meta['template']
        template = Template(template_str.replace('%(', '${').replace(')s', '}'))  # convert to Python Template

        nens = exp_meta['nens']
        ensemble_list = [f"EN{n:02d}" for n in range(1, nens + 1)]

        processor = SurfaceEnergyBalanceProcessor(
            data_dir=path,
            varnames=varnames,
            experiment=exp_name,
            ensemble_list=ensemble_list,
            years=years,
            outdir=out_path,
            lat_bounds=(-90, 90),
            lon_bounds=(0, 360),
            landmask_file=landmask_file
        )
        
        processor.process()
                   


[INFO] Running experiment: CTRLEN10
→ Processing CTRLEN10 | EN01
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[CTRLEN10] EN01 - GLOBAL: saved monthly + per-month daily files
[CTRLEN10] EN01 - LAND: saved monthly + per-month daily files
[CTRLEN10] EN01 - OCEAN: saved monthly + per-month daily files
→ Processing CTRLEN10 | EN02
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[CTRLEN10] EN02 - GLOBAL: saved monthly + per-month daily files
[CTRLEN10] EN02 - LAND: saved monthl

[INFO] Resampling 6-hourly data to daily means...
[CAPTEN10] EN06 - GLOBAL: saved monthly + per-month daily files
[CAPTEN10] EN06 - LAND: saved monthly + per-month daily files
[CAPTEN10] EN06 - OCEAN: saved monthly + per-month daily files
→ Processing CAPTEN10 | EN07
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[CAPTEN10] EN07 - GLOBAL: saved monthly + per-month daily files
[CAPTEN10] EN07 - LAND: saved monthly + per-month daily files
[CAPTEN10] EN07 - OCEAN: saved monthly + per-month daily files
→ Processing CAPTEN10 | EN08
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS 

[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[DARTEN20] EN12 - GLOBAL: saved monthly + per-month daily files
[DARTEN20] EN12 - LAND: saved monthly + per-month daily files
[DARTEN20] EN12 - OCEAN: saved monthly + per-month daily files
→ Processing DARTEN20 | EN13
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[DARTEN20] EN13 - GLOBAL: saved monthly + per-month daily files
[DARTEN20] EN13 - LAND: saved monthly + per-month daily files
[DARTEN20] EN13 - OCEAN: saved monthly +

[DARTEN40] EN07 - GLOBAL: saved monthly + per-month daily files
[DARTEN40] EN07 - LAND: saved monthly + per-month daily files
[DARTEN40] EN07 - OCEAN: saved monthly + per-month daily files
→ Processing DARTEN40 | EN08
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[DARTEN40] EN08 - GLOBAL: saved monthly + per-month daily files
[DARTEN40] EN08 - LAND: saved monthly + per-month daily files
[DARTEN40] EN08 - OCEAN: saved monthly + per-month daily files
→ Processing DARTEN40 | EN09
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied

[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[DARTEN40] EN23 - GLOBAL: saved monthly + per-month daily files
[DARTEN40] EN23 - LAND: saved monthly + per-month daily files
[DARTEN40] EN23 - OCEAN: saved monthly + per-month daily files
→ Processing DARTEN40 | EN24
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[DARTEN40] EN24 - GLOBAL: saved monthly + per-month daily files
[DARTEN40] EN24 - LAND: saved monthly + per-month daily files
[DARTEN40] EN24 - OCEAN: saved monthly + per-month daily files
→ Processing DARTEN40 | EN25
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX 

[DARTEN40] EN38 - OCEAN: saved monthly + per-month daily files
→ Processing DARTEN40 | EN39
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[DARTEN40] EN39 - GLOBAL: saved monthly + per-month daily files
[DARTEN40] EN39 - LAND: saved monthly + per-month daily files
[DARTEN40] EN39 - OCEAN: saved monthly + per-month daily files
→ Processing DARTEN40 | EN40
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] SHFLX found in 6-hourly path: resampling will be applied.
[INFO] FSNS found in 6-hourly path: resampling will be applied.
[INFO] FLNS found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[DARTEN40] EN40 - GLOBAL: saved monthly + per-month daily files
[DARTEN40]